# HeFESTo Primer

In [ ]:
import os, sys
import numpy as np
from pathlib import Path
from shutil import rmtree, copy
from matplotlib import pyplot as plt
from matplotlib import gridspec, cm

# Derive the root of this package
package_root = Path.cwd().resolve().parents[2]

HaMaGeoLib_DIR = str(package_root)
if os.path.abspath(HaMaGeoLib_DIR) not in sys.path:
    sys.path.append(os.path.abspath(HaMaGeoLib_DIR))

import hamageolib.utils.plot_helper as plot_helper
from hamageolib.utils.hefesto_helper import LOOKUP_TABLE, HEFESTO_OPT, DistributeParallelControl, convert_mol_fraction

RESULT_DIR = os.path.join(HaMaGeoLib_DIR, 'results')

## HeFESTo cases for parallel running

Below we use a dict object to specify how to distribute the HeFESTo case:
* Assign slurm file and control file
* Assign P, T ranges and intervals
* Assign number of processors

The new case contains multiple subfolders, one for a separate processer

### Generate and Run

To convert oxide mol fraction to atom mol fraction:
* Entries are mol% composition in order [SiO2, MgO, FeO, CaO, Al2O3, Na2O], and they must sum to ~100. These are how we describe compositions like pyrolite or hazburgite
* Outputs are mol% of atoms in order of [Si, Mg, Fe, Ca, Al, Na], these are inputs into HeFestTo

The number to use for basalt, harzbugite and pyrolite is from the Xu 2008 paper
* For Harzburgite, I add a 0.01 % to the Na2O mol fraction instead of directly using 0.0, which seems to produce error

The number for upper / lower crust is from the Rudnick_Gao-crust_compositionTreatise_on_Geophysics-Elsevier_2009.
* As there are K2O contents, the idea is to add it to the Na2O contents.
* Upper vs Lower Crust Composition (with K₂O)

### Upper vs Lower Crust Composition (with K₂O)

### Upper vs Lower Crust Composition (with K₂O, reordered)

| Oxide  | Molar Mass (g/mol) | UCC Mass Frac | UCC Mol Frac | LCC Mass Frac | LCC Mol Frac |
|--------|--------------------|---------------|--------------|---------------|--------------|
| SiO₂   | 60.0900            | 0.6600        | 0.7280       | 0.5400        | 0.6140       |
| MgO    | 40.3000            | 0.0250        | 0.0400       | 0.0700        | 0.0980       |
| FeO    | 71.8500            | 0.0500        | 0.0300       | 0.0900        | 0.0630       |
| CaO    | 56.0800            | 0.0350        | 0.0330       | 0.0900        | 0.0890       |
| Al₂O₃  | 101.9600           | 0.1500        | 0.0950       | 0.1600        | 0.1140       |
| Na₂O   | 61.9800            | 0.0350        | 0.0370       | 0.0250        | 0.0280       |
| K₂O    | 94.2000            | 0.0250        | 0.0170       | 0.0100        | 0.0070       |
| **Sum** | —                  | **0.9800**    | **0.9800**   | **0.9850**    | **0.9850**   |

In [ ]:
generate_case_file = False

if generate_case_file:

    composition = "harzburgite" # pyrolite, harzburgite, basalt, upper_crust, lower_crust


    option_dict = {
        "HeFESTo repository": "/quobyte/billengrp/lochy/Software/HeFESToRepository",
        "slurm path": "${HaMaGeoLib_DIR}/scripts/server/hive/job_hive_collision_2d.slurm",
        "prm directory": "/quobyte/billengrp/lochy/Software/HeFESTo_Parameters_010123",
        "control path": "${HaMaGeoLib_DIR}/hamageolib/research/haoyuan_collision0/files/HeFESTo/control",
        "output directory": "/mnt/lochy/ASPECT_DATA/HeFESTo",
        "case name": "/mnt/lochy/ASPECT_DATA/Collision0/HeFesto_Compare/hefesto_%s_P_T" % composition,
        "nproc": 32,
        "split by": "T",
        "T":
        {
            "variable": "temperature",
            "T1": 250.0,
            "T2": 4000.0,
            "nT": 256
        },
        "P":
        {
            "P1": 0.0,
            "P2": 150.0,
            "nP": 2560
        }
    }

In [ ]:
if generate_case_file:

    import json

    if composition == "pyrolite": 
        comps = [38.71, 49.85, 6.17, 2.94, 2.22, 0.11] # pyrolite
    elif composition == "harzburgite":
        comps = [36.07, 56.51, 6.07, 0.81, 0.52, 0.01] # harzburgite
    elif composition == "basalt":
        comps = [51.75, 14.94, 7.06, 13.88, 10.19, 2.18] # basalt
    elif composition == "upper_crust":
        comps = np.array([72.80, 4.00, 3.00, 3.30, 9.50, 3.70+1.70]) / 0.9800
    elif composition == "lower_crust":
        comps = np.array([61.40, 9.80, 6.30, 8.90, 11.40, 2.80+0.70]) / 0.9850
    else:
        raise ValueError("Wrong type of composition")

    comps_atom = convert_mol_fraction(comps)

    HeFESTo_Opt = HEFESTO_OPT()
    # read in json options
    HeFESTo_Opt.import_options(option_dict)
    
    # todo_table
    DistributeParallelControl(*HeFESTo_Opt.to_distribute_parallel_control(), 
                              modules=[],
                              sources=["/home/lochy/module/hefesto.sh"],
                              composition=comps_atom)

    # copy the json file
    case_dir = HeFESTo_Opt.case_dir()
    json_o_path = os.path.join(case_dir, "case.json")
    with open(json_o_path, 'w') as fout:
        json.dump(option_dict, fout)

## Plot profiles from HeFESTO & Perplex outputs

In this cookbook, the profile outputs from HeFESTO is plotted. THe outpus must be generated with a single T / S and depth-varing pressures.

Some notes on the values:
* The "thermal expansivity" and "Isobaric_heat_capacity" are effective values that has the phase transition component in it.

In [ ]:
plot_output_profile = False

if plot_output_profile:

    # assign path of the inputs
    fort56_path = "/mnt/lochy/ASPECT_DATA/Collision0/HeFesto_Compare/HeFESTo_benchmark/fort.56"
    o_dir = os.path.join(HaMaGeoLib_DIR, "HeFESTo_profile")
    if not os.path.isdir(o_dir):
        os.mkdir(o_dir)

    # read fort.56 file
    LookupTable = LOOKUP_TABLE()
    LookupTable.ReadRawFort56(fort56_path)
    depths_0, Ts_0 = LookupTable.export_temperature_profile()
    _, Ps_0 = LookupTable.export_pressure_profile()
    _, densities_0 = LookupTable.export_density_profile()


In [ ]:
if plot_output_profile:

    from matplotlib import gridspec, rcdefaults 
    from matplotlib.ticker import MultipleLocator
    from hamageolib.utils.plot_helper import scale_matplotlib_params

    depths_limit = [0, 1000.0] # km
    depth_tick = 200.0 # km

    # Retrieve the default color cycle
    default_colors = [color['color'] for color in plt.rcParams['axes.prop_cycle']]

    # Scaling parameters for plots.
    scaling_factor = 1.6  # General scaling factor for the plot size.
    font_scaling_multiplier = 1.5  # Extra scaling for fonts.
    legend_font_scaling_multiplier = 0.5  # Scaling for legend fonts.
    line_width_scaling_multiplier = 2.0  # Extra scaling for line widths
    n_minor_ticks = 4  # number of minor ticks between two major ones


    # Scale matplotlib parameters based on specified factors.
    scale_matplotlib_params(
        scaling_factor, 
        font_scaling_multiplier=font_scaling_multiplier,
        legend_font_scaling_multiplier=legend_font_scaling_multiplier,
        line_width_scaling_multiplier=line_width_scaling_multiplier
    )

    # Update font settings for compatibility with publishing tools like Illustrator.
    plt.rcParams.update({
        'font.family': 'Times New Roman',
        'pdf.fonttype': 42,
        'ps.fonttype': 42
    })
    
    # start plots
    fig = plt.figure(tight_layout=True, figsize=(18, 18))
    gs = gridspec.GridSpec(3, 3)

    # plot pressure
    ax = fig.add_subplot(gs[0, 0])
    ax.plot(Ps_0, depths_0, "-", label="HeFesto pressure", color=default_colors[2])
    
    # ax.set_xlim([1500.0, 2000.0])
    ax.set_xlabel("Pressure (GPa)")
    # x_tick_interval = 100
    # ax.xaxis.set_major_locator(MultipleLocator(x_tick_interval))
    # ax.xaxis.set_minor_locator(MultipleLocator(x_tick_interval/(n_minor_ticks+1)))

    ax.set_ylim(depths_limit)
    ax.invert_yaxis()
    ax.set_ylabel("Depth [km]")
    y_tick_interval = depth_tick
    ax.yaxis.set_major_locator(MultipleLocator(y_tick_interval))
    ax.yaxis.set_minor_locator(MultipleLocator(y_tick_interval/(n_minor_ticks+1)))

    ax.grid()

    # plot temperature
    ax = fig.add_subplot(gs[0, 1])
    ax.plot(Ts_0, depths_0, "-", label="HeFesto temperature", color=default_colors[1])
    
    # ax.set_xlim([1500.0, 2000.0])
    ax.set_xlabel("Temperature (K)")
    # x_tick_interval = 100
    # ax.xaxis.set_major_locator(MultipleLocator(x_tick_interval))
    # ax.xaxis.set_minor_locator(MultipleLocator(x_tick_interval/(n_minor_ticks+1)))

    ax.set_ylim(depths_limit)
    ax.invert_yaxis()
    ax.set_ylabel("Depth [km]")
    y_tick_interval = depth_tick
    ax.yaxis.set_major_locator(MultipleLocator(y_tick_interval))
    ax.yaxis.set_minor_locator(MultipleLocator(y_tick_interval/(n_minor_ticks+1)))

    ax.grid()

    # plot density
    ax = fig.add_subplot(gs[0, 2])
    ax.plot(densities_0, depths_0, "-", label="HeFesto density", color=default_colors[0])
    
    # ax.set_xlim([1500.0, 2000.0])
    ax.set_xlabel("Density (kg/m^3)")
    # x_tick_interval = 100
    # ax.xaxis.set_major_locator(MultipleLocator(x_tick_interval))
    # ax.xaxis.set_minor_locator(MultipleLocator(x_tick_interval/(n_minor_ticks+1)))

    ax.set_ylim(depths_limit)
    ax.invert_yaxis()
    ax.set_ylabel("Depth [km]")
    y_tick_interval = depth_tick
    ax.yaxis.set_major_locator(MultipleLocator(y_tick_interval))
    ax.yaxis.set_minor_locator(MultipleLocator(y_tick_interval/(n_minor_ticks+1)))

    ax.grid()

    # plot vs
    field = "VS"
    _, data_0 = LookupTable.export_field_profile(field)

    ax = fig.add_subplot(gs[1, 0])
    ax.plot(data_0, depths_0, "-", label="HeFesto %s" % field, color=default_colors[0])
    
    # ax.set_xlim([1500.0, 2000.0])
    ax.set_xlabel("VS (m/s)")
    # x_tick_interval = 100
    # ax.xaxis.set_major_locator(MultipleLocator(x_tick_interval))
    # ax.xaxis.set_minor_locator(MultipleLocator(x_tick_interval/(n_minor_ticks+1)))

    ax.set_ylim(depths_limit)
    ax.invert_yaxis()
    ax.set_ylabel("Depth [km]")
    y_tick_interval = depth_tick
    ax.yaxis.set_major_locator(MultipleLocator(y_tick_interval))
    ax.yaxis.set_minor_locator(MultipleLocator(y_tick_interval/(n_minor_ticks+1)))

    ax.grid()

    # plot vp
    field = "VP"
    _, data_0 = LookupTable.export_field_profile(field)

    ax = fig.add_subplot(gs[1, 1])
    ax.plot(data_0, depths_0, "-", label="HeFesto %s" % field, color=default_colors[0])
    
    # ax.set_xlim([1500.0, 2000.0])
    ax.set_xlabel("VP (m/s)")
    # x_tick_interval = 100
    # ax.xaxis.set_major_locator(MultipleLocator(x_tick_interval))
    # ax.xaxis.set_minor_locator(MultipleLocator(x_tick_interval/(n_minor_ticks+1)))

    ax.set_ylim(depths_limit)
    ax.invert_yaxis()
    ax.set_ylabel("Depth [km]")
    y_tick_interval = depth_tick
    ax.yaxis.set_major_locator(MultipleLocator(y_tick_interval))
    ax.yaxis.set_minor_locator(MultipleLocator(y_tick_interval/(n_minor_ticks+1)))

    ax.grid()

    # plot thermal expansivity
    field = "Thermal_expansivity"
    _, data_0 = LookupTable.export_field_profile(field)

    ax = fig.add_subplot(gs[2, 0])
    ax.plot(data_0, depths_0, "-", label="HeFesto %s" % field, color=default_colors[0])

    # ax.set_xlim([1.0, 2.0])
    ax.set_xlabel("Thermal_expansivity (k^-1)")
    # x_tick_interval = 0.2
    # ax.xaxis.set_major_locator(MultipleLocator(x_tick_interval))
    # ax.xaxis.set_minor_locator(MultipleLocator(x_tick_interval/(n_minor_ticks+1)))
    
    ax.set_ylim(depths_limit)
    ax.invert_yaxis()
    ax.set_ylabel("Depth [km]")
    y_tick_interval = depth_tick
    ax.yaxis.set_major_locator(MultipleLocator(y_tick_interval))
    ax.yaxis.set_minor_locator(MultipleLocator(y_tick_interval/(n_minor_ticks+1)))

    ax.grid()

    # plot heat capacity 
    field = "Isobaric_heat_capacity"
    _, data_0 = LookupTable.export_field_profile(field)
    
    ax = fig.add_subplot(gs[2, 1])
    ax.plot(data_0, depths_0, "-", label="HeFesto %s" % field, color=default_colors[0])
   
    ax.set_xlim([1.0, 2.0])
    ax.set_xlabel("Isobaric_heat_capacity (J/Kg/K)")
    x_tick_interval = 0.2
    ax.xaxis.set_major_locator(MultipleLocator(x_tick_interval))
    ax.xaxis.set_minor_locator(MultipleLocator(x_tick_interval/(n_minor_ticks+1)))
    
    ax.set_ylim(depths_limit)
    ax.invert_yaxis()
    ax.set_ylabel("Depth [km]")
    y_tick_interval = depth_tick
    ax.yaxis.set_major_locator(MultipleLocator(y_tick_interval))
    ax.yaxis.set_minor_locator(MultipleLocator(y_tick_interval/(n_minor_ticks+1)))

    ax.grid()

    # save plot
    o_path = os.path.join(RESULT_DIR, "HeFESTo_profile.png")
    fig.savefig(o_path)
    print("generate figure %s" % o_path)

    plt.close()
    
    rcdefaults()

## Combine HeFESTo outputs to Perple_X style lookup table

In [ ]:
export_perplex_table = True

if export_perplex_table:
    case_dir = "/mnt/lochy/ASPECT_DATA/Collision0/HeFesto_Compare/hefesto_upper_crust_P_T"

### First assemble outputs from parallel runs

The HeFESTo code is parallel with slurm, so the fort.56 files are distributed by the threads.
The first step is to find all the fort.56 outputs and combine them into a collective one.

The output file will be automatically saved to this location:

    output/fort.56

In [ ]:
if export_perplex_table:
    from hamageolib.utils.hefesto_helper import AssembleParallelFiles
    
    AssembleParallelFiles(case_dir)

### Generate table for ASPECT

Before the table is ready for ASPECT, we check that the data aligned in the two dimentions and then we properly add a header to the table

This might not work at first try.
The outputs from HeFESTo could be messed up if there is a long string, like this example:

    3.5274338280952606-2782772.4668332533910871

We need to first add a blank in between:

    3.5274338280952606 -2782772.4668332533910871

The options of 

    exchange_dimension=True

would exchange the 1st and 2nd dimensions of the output. In this case, I use it to first output the temperature

In [ ]:
if export_perplex_table:

    # generate table
    
    input_file = os.path.join(case_dir, "output/fort.56")
    
    # use part of the data
    # interval1=10; interval2=11
    # output_file = os.path.join(case_dir, "output/table_PT_small.txt")
    
    # use all data 
    interval1=1; interval2=1
    output_file = os.path.join(case_dir, "output/table_PT.txt")
    
    exchange_dimension=False
    
    # assert input file exists
    assert(os.path.isfile(input_file))
    
    if (os.path.isfile(output_file)):  # remove old files
        os.remove(output_file)
    
    # read input
    LookupTable = LOOKUP_TABLE()
    LookupTable.ReadRawFort56(input_file)
    
    # process output
    field_names = ['Pressure', 'Temperature', "Entropy", 'Density', 'Thermal_expansivity', 'Isobaric_heat_capacity',\
                    'VP', 'VS', 'Enthalpy', "most abundant phase"]
    maph_lookup = ["pv"]
    LookupTable.Process(field_names, output_file, interval1=interval1, interval2=interval2, file_type="structured",\
                        fix_coordinate_minor=True, exchange_dimension=exchange_dimension, maph_lookup=maph_lookup)
    
    # assert something 
    assert(os.path.isfile(output_file))

## Inspect a table

In [ ]:
inspect_lookup_table = True

if inspect_lookup_table:

    # Inherite the table generated in the last step
    table_path = output_file 
     
    # case_dir = "/mnt/lochy/ASPECT_DATA/Collision0/HeFesto_Compare/hefesto_pyrolite_P_T"
    # table_path = os.path.join(case_dir, "output/table_PT_small.txt")

    LookupTable = LOOKUP_TABLE()
    LookupTable.ReadPerplex(table_path, header_rows=4)
    LookupTable.Update()
    print(LookupTable.AllFields())

    # assert this because the following function will extract this in order
    assert(LookupTable.first_dimension_name == "Pressure") 

    Ps, Ts, Densities = LookupTable.export_field_mesh("Density")
    Ps, Ts, Alphas = LookupTable.export_field_mesh("Thermal_expansivity")

In [ ]:
if inspect_lookup_table:
    
    # Retrieve the default color cycle
    from matplotlib.ticker import MultipleLocator
    from matplotlib import rcdefaults
    default_colors = [color['color'] for color in plt.rcParams['axes.prop_cycle']]
    
    # Example usage
    # Rule of thumbs:
    # 1. Set the limit to something like 5.0, 10.0 or 50.0, 100.0 
    # 2. Set five major ticks for each axis
    scaling_factor = 1.0  # scale factor of plot
    font_scaling_multiplier = 1.5 # extra scaling multiplier for font
    legend_font_scaling_multiplier = 0.5
    line_width_scaling_multiplier = 2.0 # extra scaling multiplier for lines
    
    T_lim = (400.0, 2000.0) # T (C)
    T_level = 50  # number of levels in contourf plot
    T_tick_interval = 200.0  # tick interval along v
    
    P_lim = (0.0, 50.0) # P (Gpa)
    P_level = 50  # number of levels in contourf plot
    P_tick_interval = 10.0 # tick interval along v
    
    n_minor_ticks = 4  # number of minor ticks between two major ones
    
    # scale the matplotlib params
    plot_helper.scale_matplotlib_params(scaling_factor, font_scaling_multiplier=font_scaling_multiplier,\
                            legend_font_scaling_multiplier=legend_font_scaling_multiplier,
                            line_width_scaling_multiplier=line_width_scaling_multiplier)
    

    # Update font settings for compatibility with publishing tools like Illustrator.
    plt.rcParams.update({
        'font.family': 'Times New Roman',
        'pdf.fonttype': 42,
        'ps.fonttype': 42
    })

    fig = plt.figure(tight_layout=True, figsize=(12, 5))
    gs = gridspec.GridSpec(1, 2)

    # density
    ax = fig.add_subplot(gs[0, 0])
    h = ax.pcolormesh(Ts, Ps/1e4, Densities, vmin=3000.0, vmax=5000.0)
    # ax.contour(Ts-273.15, Ps/1e4, Densities, (10, 20, 40), cmap="Greys", linestyles="dashed")
    fig.colorbar(h, ax=ax, label="Density (kg/m^3)")

    ax.set_xlim(T_lim)
    ax.set_xlabel(r"T (K)")
    ax.xaxis.set_major_locator(MultipleLocator(T_tick_interval))
    ax.xaxis.set_minor_locator(MultipleLocator(T_tick_interval/(n_minor_ticks+1)))

    ax.set_ylim(P_lim)
    ax.set_ylabel("P (GPa)")
    ax.yaxis.set_major_locator(MultipleLocator(P_tick_interval))
    ax.yaxis.set_minor_locator(MultipleLocator(P_tick_interval/(n_minor_ticks+1)))
    ax.invert_yaxis()
    
    # Adjust spine thickness for this plot
    for spine in ax.spines.values():
        spine.set_linewidth(0.5 * scaling_factor * line_width_scaling_multiplier)

    
    # thermal expansivity
    ax = fig.add_subplot(gs[0, 1])
    h = ax.pcolormesh(Ts, Ps/1e4, Alphas, vmin=1e-5, vmax=1e-4)
    # ax.contour(Ts-273.15, Ps/1e4, Densities, (10, 20, 40), cmap="Greys", linestyles="dashed")
    fig.colorbar(h, ax=ax, label="Thermal Expansivity (1/K)")

    ax.set_xlim(T_lim)
    ax.set_xlabel(r"T (K)")
    ax.xaxis.set_major_locator(MultipleLocator(T_tick_interval))
    ax.xaxis.set_minor_locator(MultipleLocator(T_tick_interval/(n_minor_ticks+1)))

    ax.set_ylim(P_lim)
    ax.set_ylabel("P (GPa)")
    ax.yaxis.set_major_locator(MultipleLocator(P_tick_interval))
    ax.yaxis.set_minor_locator(MultipleLocator(P_tick_interval/(n_minor_ticks+1)))
    ax.invert_yaxis()
    
    # Adjust spine thickness for this plot
    for spine in ax.spines.values():
        spine.set_linewidth(0.5 * scaling_factor * line_width_scaling_multiplier)
